# 머신러닝 실습

**Machine Learning · ML · 기계학습**

데이터에서 패턴을 학습해 새로운 입력의 값을 예측하거나 분류하는 방법.

소재 분야에서 이해하기: 조성과 공정 조건으로 소재의 강도를 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 데이터 만들기

조성·공정 조건에서 경도를 예측하는 문제를 흉내낸 합성 데이터를 씁니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

## 2. 학습과 예측

데이터를 학습용과 시험용으로 나누고, 학습용만 보고 규칙을 학습합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)
pred = model.predict(X_test)
print('시험 데이터 MAE %.2f HV' % mean_absolute_error(y_test, pred))

plt.scatter(y_test, pred, s=18)
lo, hi = y_test.min(), y_test.max()
plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
plt.xlabel('measured hardness (HV)'); plt.ylabel('predicted hardness (HV)'); plt.title('predicted vs measured')
plt.show()

## 3. 해석

대각선에 가까울수록 예측이 잘 맞은 것입니다. 모델은 학습 데이터에서 본 조건 범위 안에서만
신뢰할 수 있고, 정답을 모르는 새 조건에서는 오차가 커질 수 있습니다.

In [ ]:
# 학습 데이터에 없던 범위(소성온도 1000 C)를 넣으면 어떻게 되는지 확인합니다.
outside = np.array([[1000.0, 4.0, 2.0, 0.0]])
print('학습 범위 최대 소성온도 %.0f C' % X[:, 0].max())
print('1000 C 예측값 %.1f HV (범위 밖이므로 신뢰할 수 없습니다)' % model.predict(outside)[0])

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#ml)을 여세요.